[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_89_Agentic_RAG_Multi_Hop_Retrieval.ipynb)

# Lesson 89 — Agentic RAG: Adaptive, Multi-Hop Retrieval
### Phase 10 · RAG at Production Scale (Lesson 7 of the phase)

**Where we are.** Across Phase 10 you rebuilt the *single-shot* RAG pipeline the way it actually
ships — one question in, one retrieval, one answer out:

| # | Lesson | The lever it pulls |
|---|--------|--------------------|
| 83 | Production baseline + the four failure modes | *problem framing & measurement* |
| 84 | Chunking strategies | how you **cut** the docs |
| 85 | Dense & hybrid retrieval | how you **match** query → doc |
| 86 | Cross-encoder reranking | how you **reorder** candidates |
| 87 | Grounding, citations & abstention | how you **answer** (or refuse) |
| 88 | Query transformation & rewriting | how you **rephrase** the query *before* it hits the index |
| **89** | **Agentic RAG (today)** | how you **loop** — retrieve, judge, retrieve again |

**The ceiling we hit.** Everything so far assumes **one retrieval is enough**. It isn't, for a huge
class of real questions — most sharply the **multi-hop / bridge** question:

> *"How much water does the Barista Pro hold?"* — when **no single passage** says both "Barista Pro"
> **and** the capacity. You must first learn *which boiler* the Barista Pro uses, **then** look up
> *that boiler's* capacity. One retrieval structurally cannot chain those two facts.

Today we add the missing piece: a **control loop** that sits on top of the retriever you already
built and decides *whether to retrieve, what to retrieve next, and when to stop.* This is where
"RAG" becomes an **agent** — the direct heir of the ReAct loop you wrote back in Lesson 4, now
pointed at **retrieval** instead of tools.

> **One idea to keep:** *single-shot RAG retrieves once and hopes. Agentic RAG retrieves, **grades
> its own evidence**, and keeps going until it can actually answer — under a strict hop budget.*


## 0 · Setup

Run this first. **You do not need an API key.** If an `ANTHROPIC_API_KEY` is found in Colab Secrets
(🔑 in the left sidebar), the agent's judgments (grade, next-hop, final answer) are made by a **real
LLM**. If not, we fall back to **transparent, deterministic heuristics** so every cell runs and
every number below is reproducible. The lesson is about the **control flow**, not the cleverness of
any single call — the mock is enough to *see the mechanism*, and a real key upgrades every decision
for free.

In [ ]:
# Only stdlib + a little math. No downloads, no vector DB — the retriever is deliberately
# transparent so you can SEE why each hop matches (same spirit as L83/L88).
import os, re, math
from collections import Counter, defaultdict
from dataclasses import dataclass

# --- real-or-mock LLM ---------------------------------------------------------
LLM = None
try:
    import anthropic
    key = None
    try:
        from google.colab import userdata          # Colab
        key = userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        key = os.environ.get("ANTHROPIC_API_KEY")   # local
    if key:
        LLM = anthropic.Anthropic(api_key=key)
except Exception:
    LLM = None

LLM_AVAILABLE = LLM is not None
MODEL = "claude-haiku-4-5-20251001"   # cheap + fast is perfect for the loop's inner judgments

def call_llm(prompt, max_tokens=200):
    # Real Anthropic call when a key exists; otherwise raises so callers use their fallback.
    if not LLM_AVAILABLE:
        raise RuntimeError("no LLM")
    m = LLM.messages.create(model=MODEL, max_tokens=max_tokens,
                            messages=[{"role": "user", "content": prompt}])
    return m.content[0].text.strip()

print("LLM mode :", "REAL (Anthropic)" if LLM_AVAILABLE else "MOCK (deterministic heuristics)")
print("Everything below runs identically in either mode — only answer *quality* changes.")


## 1 · A knowledge base built for hops

A tiny home-espresso KB, but arranged so **key facts live one hop apart**. Notice the shape: the
passage that states a boiler's *capacity* names only the **boiler** — never the **machine**. To
answer a question about a *machine*, you must first retrieve the passage that links
`machine → boiler`. That link is the **bridge**. There are also **distractor** passages that share
words with the questions but don't answer them — exactly what makes a single-shot retriever grab the
wrong thing.

In [ ]:
KB = {
    # --- Barista Pro chain: machine -> Titan-X boiler -> capacity ---
    "p1":  "The Barista Pro machine contains the Titan-X boiler.",   # BRIDGE (machine -> boiler)
    "p2":  "The Titan-X boiler stores 1.5 litres.",                  # ANSWER (names boiler, not machine)
    # --- Home Cafe chain ---
    "p3":  "The Home Cafe machine contains the AquaCore boiler.",    # BRIDGE
    "p4":  "The AquaCore boiler stores 2.0 litres.",                 # ANSWER
    # --- Nimbus chain ---
    "p5":  "The Nimbus machine contains the NanoHeat boiler.",       # BRIDGE
    "p6":  "The NanoHeat boiler stores 0.8 litres.",                 # ANSWER
    # --- distractors (share words with questions, but don't answer them) ---
    "p7":  "The Barista Pro has a removable drip tray.",             # DISTRACTOR
    "p8":  "The Home Cafe includes a cup warmer.",                   # DISTRACTOR
    # --- direct single-hop facts (for the router + eval) ---
    "p9":  "Espresso tastes sour when the grind is too coarse.",
    "p10": "The Barista Pro has a 54mm portafilter.",
}
print(f"{len(KB)} passages:")
for pid, txt in KB.items():
    print(f"  {pid:>3}: {txt}")


### 1.1 A transparent lexical retriever

We score a passage by the **sum of IDF weights of the query terms it contains** — a BM25-style
lexical score. It's fully offline, deterministic, and lets you *read* exactly why a passage matched:
rare, informative words (like `titan-x`, `portafilter`) count for a lot; common words count for
little. The multi-hop *mechanism* we build today is retriever-agnostic — swap in a dense retriever
from L85 and it works for the same reason (each hop enriches the query's overlap with the *next*
passage in the chain).

In [ ]:
STOP = set("a an the of to in on for and or is are be with your you my i it its this that too "
               "how much does do can what when where why if used use uses have has need needs".split())

def tokens(text):
    # words; keep internal dots/hyphens (1.5, titan-x) but DROP trailing punctuation
    return [w for w in re.findall(r"[a-z0-9]+(?:[.\-][a-z0-9]+)*", text.lower())
            if w not in STOP and len(w) > 1]

DF = Counter()
for txt in KB.values():
    for t in set(tokens(txt)):
        DF[t] += 1
N = len(KB)
IDF = {t: math.log((N + 1) / (df + 0.5)) for t, df in DF.items()}
def idf(t): return IDF.get(t, math.log((N + 1) / 0.5))   # unseen query words still get weight

def score(query, doc):
    qs = set(tokens(query))
    return sum(idf(t) for t in set(tokens(doc)) if t in qs)

def retrieve(query, k=3, exclude=None):
    exclude = exclude or set()
    scored = [(pid, score(query, txt), txt) for pid, txt in KB.items() if pid not in exclude]
    scored.sort(key=lambda r: (r[1], r[0]), reverse=True)   # score, then pid for a stable order
    return scored[:k]

# 💡 EXPERIMENT: retrieve("Titan-X boiler capacity") and watch p2 jump to the top.
print("retrieve('how much water does the Barista Pro hold'):")
for pid, s, txt in retrieve("how much water does the Barista Pro hold", k=3):
    print(f"  {pid}  {s:.2f}  {txt}")


## 2 · Watch single-shot RAG **fail** the bridge question

The question: **"What is the water capacity of the Barista Pro?"** The gold passage is **p2** —
*"The Titan-X boiler stores 1.5 litres."* But look at p2's words vs the question's: they share
**nothing** (`titan-x, boiler, stores, litres` vs `water, capacity, barista, pro`). No single-shot
retriever — lexical *or* dense — can jump straight to p2, because the question and the answer share
no common ground. The only route is *through* p1 (Barista Pro → Titan-X boiler). This is the
defining shape of a **multi-hop** question.

In [ ]:
BRIDGE_Q = "What is the water capacity of the Barista Pro?"
GOLD = "p2"

hits = retrieve(BRIDGE_Q, k=3)
print("Single-shot top-3 for:", BRIDGE_Q, "\n")
for rank, (pid, s, txt) in enumerate(hits, 1):
    print(f"  #{rank}  {pid}  {s:.2f}  {txt}{'   <-- GOLD' if pid == GOLD else ''}")
print(f"\nGold {GOLD} score vs the question: {score(BRIDGE_Q, KB[GOLD]):.2f}  (zero shared words)")
print(f"Gold {GOLD} in single-shot top-3?  {GOLD in [p for p,_,_ in hits]}")
print("=> The generator never even SEES the answer. This is a retrieval miss, not a prompt bug.")


## 3 · The agentic RAG loop — the concept

Instead of *retrieve → generate*, we **loop**:

```
        +-----------------------------------------------------------+
        |                                                           v
  question --> RETRIEVE top-k --> add to evidence --> GRADE evidence -- SUFFICIENT? --> GENERATE grounded answer
        ^                                                           |
        |                                                           | INSUFFICIENT
        +----------------- PROPOSE next-hop query <-----------------+
                          (fold in the bridge terms we just learned)
                                     (bounded by a HOP BUDGET)
```

Three LLM-shaped judgments turn a dumb retriever into an agent:

1. **GRADE** — *"Given the question and the evidence so far, can I answer yet?"* The loop's brain.
2. **PROPOSE NEXT-HOP** — *"What should I look up next?"* We mine **bridge terms** — informative
   words that appeared in the retrieved passages but weren't in the original question (e.g.
   `titan-x`) — and fold them into the next query.
3. **GENERATE** — answer **only** from the collected evidence, citing passage ids (your L87
   grounding discipline, unchanged).

The **hop budget** is non-negotiable: without it, a confused agent retrieves forever. Cap it (2–4
hops) and treat "hit the cap without sufficiency" as a controlled abstention.

In [ ]:
def bridge_terms(question, evidence_texts, used=None, top=3):
    # Informative words in the evidence but NOT in the question — the entities we just discovered.
    # Keep only DF>=2 (they link to other passages) AND specific (high idf, so we chase entities
    # like 'titan-x', not generic words like 'boiler'). Skip ones we already searched.
    used = used or set()
    q = set(tokens(question))
    cand = {}
    for txt in evidence_texts:
        for t in tokens(txt):
            if (t not in q and DF[t] >= 2 and idf(t) >= 1.3
                    and not re.fullmatch(r"[0-9.]+", t) and t not in used):
                cand[t] = idf(t)
    return sorted(cand, key=lambda t: (cand[t], t), reverse=True)[:top]

def best_hop(question, evidence, used):
    # Look-ahead: among candidate bridge terms, pick the one that retrieves the best UNSEEN passage.
    seen, best = set(evidence), None
    for t in bridge_terms(question, [evidence[p] for p in evidence], used):
        hit = retrieve(question + " " + t, k=1, exclude=seen)
        if hit and hit[0][1] > 0 and (best is None or hit[0][1] > best[1]):
            best = (t, hit[0][1])
    return best   # (term, score) or None

print("bridge terms after seeing only p1:", bridge_terms(BRIDGE_Q, [KB["p1"]]))
print("best next hop after seeing p1    :", best_hop(BRIDGE_Q, {"p1": KB["p1"]}, set()))


### 3.1 The three judgments (real LLM, or deterministic fallback)

With a key, **grade** and **next-hop** are real LLM calls. Without one, we use transparent proxies:

- **grade** → *sufficient* when a collected passage **directly answers** the question (high lexical
  overlap) **or** there is **no productive bridge left to follow** (the chain is fully walked).
- **next-hop** → fold in the best bridge term we found.

Both proxies are deterministic, so you can watch — and reproduce — every decision.

In [ ]:
ANSWER_FLOOR = 3.5   # lexical overlap above this = a passage that directly answers the question

def grade_sufficient(question, evidence, direct_score, bh):
    if LLM_AVAILABLE:
        ev = "\n".join(f"- {t}" for t in evidence.values())
        try:
            v = call_llm(f"Question: {question}\nEvidence:\n{ev}\n\n"
                         "Can the question be fully answered using ONLY this evidence? "
                         "Reply one word: SUFFICIENT or INSUFFICIENT.", max_tokens=5)
            return v.upper().startswith("SUFF")
        except Exception:
            pass
    return direct_score >= ANSWER_FLOOR or bh is None      # deterministic proxy

def propose_next_hop(question, evidence, bh):
    if LLM_AVAILABLE:
        ev = "\n".join(f"- {t}" for t in evidence.values())
        try:
            return call_llm(f"Original question: {question}\nEvidence so far:\n{ev}\n\n"
                            "The evidence is NOT yet enough. Write ONE short follow-up search query "
                            "to find the missing fact. Reply with only the query text.", max_tokens=40)
        except Exception:
            pass
    return (question + " " + bh[0]) if bh else question    # deterministic: fold in the bridge term

def generate_answer(question, evidence, focus, landing=None):
    if not evidence:
        return "I do not have enough information to answer that.", []
    if LLM_AVAILABLE:
        ctx = "\n".join(f"[{pid}] {txt}" for pid, txt in evidence.items())
        try:
            return call_llm("Answer using ONLY the context. Cite the passage ids in square brackets. "
                            "If the answer is not present, say you don't know.\n\n"
                            f"Context:\n{ctx}\n\nQuestion: {question}\nAnswer:", max_tokens=200), \
                   list(evidence)
        except Exception:
            pass
    # grounded stub: cite the passage the FINAL hop went looking for (the freshly bridged fact),
    # falling back to the best focus-match if there was no hop.
    best = landing if landing in evidence else max(evidence, key=lambda p: score(focus, evidence[p]))
    return f"(grounded stub) Based on [{best}]: {evidence[best]}", [best]


## 4 · Put it together — the agentic RAG loop

~20 lines. Everything above is a helper; this is the agent. It returns a full **trace** so you can
audit *every* hop — non-negotiable for a production agent. Note the exits: **sufficient**, **no
productive bridge**, or **budget exhausted**.

In [ ]:
@dataclass
class Hop:
    n: int; query: str; retrieved: list; new_ids: list; verdict: str

def agentic_rag(question, k=3, max_hops=3, verbose=True):
    evidence, trace, used, focus, subq, n_ret, landing = {}, [], set(), question, question, 0, None
    for hop in range(1, max_hops + 1):
        hits = retrieve(subq, k=k, exclude=set(evidence)); n_ret += 1
        new = [p for p, s, _ in hits if s > 0 and p not in evidence]
        for p, s, t in hits:
            if s > 0: evidence[p] = t         # only accumulate passages that actually matched
        if new: landing = new[0]              # top-scoring fresh passage = what this hop found
        direct = max((score(question, evidence[p]) for p in evidence), default=0)
        bh = best_hop(question, evidence, used)
        stop = grade_sufficient(question, evidence, direct, bh)
        verdict = "SUFFICIENT -> answer" if stop else f"INSUFFICIENT -> hop on '{(bh or ['?'])[0]}'"
        trace.append(Hop(hop, subq, [p for p, _, _ in hits], new, verdict))
        if verbose:
            print(f"HOP {hop} | query: {subq!r}")
            print(f"        retrieved {[p for p,_,_ in hits]}  new={new}  -> {verdict}")
        if stop or bh is None:
            break
        used.add(bh[0]); subq = focus = propose_next_hop(question, evidence, bh)
    answer, cited = generate_answer(question, evidence, focus, landing)
    return {"answer": answer, "cited": cited, "evidence_ids": list(evidence),
            "trace": trace, "n_retrievals": n_ret}

print("=" * 74)
res = agentic_rag(BRIDGE_Q)
print("=" * 74)
print("ANSWER :", res["answer"])
print("CITED  :", res["cited"])
print(f"\nGold p2 reached? {'p2' in res['evidence_ids']}   (single-shot could NOT reach it in §2)")


**Read the trace.** Hop 1 retrieves the distractor + the **bridge** passage p1 (Barista Pro →
Titan-X boiler) but **not** the answer. The agent mines the bridge term `titan-x`, rewrites the
query, and Hop 2 pulls **p2** — the capacity fact that was *unreachable* from the original wording.
That is multi-hop retrieval: the agent used what it learned in hop 1 to ask a better question in hop
2. A single-shot pipeline structurally cannot do this.

## 5 · Corrective RAG (CRAG): grade the *retrieval*, not just the evidence

Multi-hop assumes the answer is *somewhere* in the corpus. But what if it isn't? A robust agent also
**grades retrieval quality** and takes corrective action:

- **Top score healthy** → proceed.
- **Top score weak** → the query is probably phrased badly → **rewrite once and retry** (your L88
  transformation, now triggered *conditionally*).
- **Still weak** → **abstain** (your L87 discipline) instead of hallucinating.

This is the [Corrective RAG](https://arxiv.org/abs/2401.15884) pattern in miniature. Watch it refuse
a question the KB simply doesn't cover.

In [ ]:
def corrective_rag(question, k=3, floor=0.5):
    hits = retrieve(question, k=k)
    top = hits[0][1] if hits else 0.0
    print(f"attempt 1  top-score={top:.2f}  {'OK' if top >= floor else 'WEAK'}")
    if top < floor:
        matched = [h[2] for h in hits if h[1] > 0]                 # only rewrite from real matches
        rewritten = (question + " " + " ".join(bridge_terms(question, matched))).strip()
        hits2 = retrieve(rewritten, k=k)
        top2 = hits2[0][1] if hits2 else 0.0
        print(f"rewrite -> {rewritten!r}\nattempt 2  top-score={top2:.2f}  "
              f"{'OK' if top2 >= floor else 'STILL WEAK -> ABSTAIN'}")
        if top2 < floor:
            return {"answer": "I don't have information on that in the knowledge base.",
                    "abstained": True}
        hits = hits2
    b = max(hits, key=lambda h: h[1])
    return {"answer": f"(grounded stub) [{b[0]}] {b[2]}", "abstained": False}

print("--- in-scope question ---")
print(corrective_rag("What portafilter does the Barista Pro use?")["answer"], "\n")
print("--- OUT-OF-scope question (KB has nothing about this) ---")
print(corrective_rag("What is the capital of France?")["answer"])


## 6 · The loop self-regulates — you don't pay the tax on easy questions

Agentic RAG is **more accurate but more expensive** — every hop is another retrieval (and, with a
key, more LLM calls). The good news: the **grader already keeps easy questions cheap**. When the
first retrieval *directly answers* the question, the loop stops after **one hop** — effectively
single-shot. Only genuinely multi-hop questions spend more. So the "should I use the loop?" router
is the loop itself: *simple queries terminate in 1 hop; bridge queries escalate automatically.*

This is the same instinct as L88's "don't transform every query" — now baked into the control flow.

In [ ]:
def routed_rag(question):
    res = agentic_rag(question, verbose=False)
    hops = len(res["trace"])
    return {"route": "single-shot" if hops == 1 else "agentic", "hops": hops, "answer": res["answer"]}

for q in ["What portafilter does the Barista Pro use?",   # easy   -> 1 hop
          "What is the water capacity of the Barista Pro?"]:  # bridge -> escalates
    r = routed_rag(q)
    print(f"[{r['route']:>11} | {r['hops']} hop(s)]  {q}")
    print(f"              -> {r['answer']}\n")


## 7 · Does it actually help? A labeled mini-eval

Talk is cheap — measure. We label a handful of questions with the gold passage that truly answers
them (some **single-hop**, some **multi-hop bridge**), then compare **single-shot top-1** vs
**agentic (gold reached anywhere in collected evidence)**, and report the **cost tax** (retrievals)
so the trade-off is honest.

In [ ]:
EVAL = [
    ("What is the water capacity of the Barista Pro?", "p2",  "bridge"),
    ("What is the water capacity of the Home Cafe?",   "p4",  "bridge"),
    ("What is the water capacity of the Nimbus?",      "p6",  "bridge"),
    ("What portafilter does the Barista Pro use?",     "p10", "single"),
    ("Why does espresso taste sour?",                  "p9",  "single"),
]

def single_shot_hit1(q, gold): return retrieve(q, k=1)[0][0] == gold
def agentic_eval(q, gold):
    r = agentic_rag(q, verbose=False)
    return gold in r["evidence_ids"], r["n_retrievals"]

print(f"{'kind':<7}{'single@1':<10}{'agentic':<9}{'retr':<6}question")
print("-" * 74)
ss_hits = ag_hits = ag_cost = 0
for q, gold, kind in EVAL:
    ss = single_shot_hit1(q, gold)
    ag, cost = agentic_eval(q, gold)
    ss_hits += ss; ag_hits += ag; ag_cost += cost
    print(f"{kind:<7}{str(ss):<10}{str(ag):<9}{cost:<6}{q}")
print("-" * 74)
n = len(EVAL)
print(f"single-shot hit@1 : {ss_hits}/{n} = {ss_hits/n:.0%}")
print(f"agentic gold-reach: {ag_hits}/{n} = {ag_hits/n:.0%}   "
      f"(avg {ag_cost/n:.1f} retrievals/question = the loop tax)")
print("\nThe uplift is concentrated on the BRIDGE rows — exactly the questions single-shot")
print("structurally cannot reach. On the single-hop rows agentic matches it (no loss) while")
print("terminating in 1 hop — which is why §6 needs no separate router.")


## 8 · Production notes — the parts the demo hides

1. **The hop budget is a safety fuse, not a tuning knob.** An agent that misjudges sufficiency will
   loop until you stop it. Cap hops (2–4), and log "hit the cap" as a distinct outcome you alert on.
2. **Grading is the hard part.** The loop is only as good as *"can I answer yet?"*. Use a cheap,
   fast model (Haiku) for the inner judgments and reserve a stronger model for the final answer.
   Calibrate the grader like any classifier — your L29 Brier/ECE tools apply directly.
3. **Cost amplifies multiplicatively.** k passages × H hops × judgment calls. A 3-hop loop can be
   ~6× a single-shot query. Because the grader stops easy questions after 1 hop, most traffic never
   pays that — but you must *measure* it, not assume it.
4. **Compose, don't replace.** Agentic RAG sits *on top of* everything you built: each hop should be
   `transform (L88) → hybrid retrieve (L85) → rerank (L86)`, then a grounded, cited answer (L87) at
   the end. Here each hop is a bare lexical lookup only to keep the mechanism visible.
5. **Always keep the trace.** Store every hop's query, retrieved ids, and verdict. When an answer is
   wrong you must see *which hop* went sideways — same audit discipline as L48 tracing / L82 orchestration.
6. **Know when NOT to loop.** Pure lookups and latency-critical paths should stay single-shot.
   Agentic RAG earns its cost on multi-hop, compound, and low-confidence queries — nowhere else.

### Ten agentic-RAG pitfalls

1. **No hop budget** → runaway loops and runaway cost.
2. **Grader with no abstention** → the loop declares victory on junk evidence.
3. **Re-retrieving the same passages every hop** (bad next-hop query) → convergence stalls; dedup
   and *change* the query with new bridge terms.
4. **Chasing values, not entities** — a next-hop query built from numbers (`1.5`, `2.0`) retrieves
   nothing; bridge on the *entity* (`titan-x`), and skip generic terms (`boiler`).
5. **Looping on simple queries** → pay 6× for no gain; let the grader terminate them in 1 hop.
6. **Uncalibrated grader** → systematically stops too early (misses) or too late (cost). Measure it.
7. **Losing provenance** — no per-hop trace means unauditable answers.
8. **One strong model for every inner call** — burns money; Haiku the judgments, save the big model
   for final synthesis.
9. **Treating agentic RAG as a silver bullet** — it fixes *retrieval reach*, not bad chunks (L84) or
   a weak reranker (L86). Garbage passages in, garbage answer out.
10. **No cap on total corpus reads** — a hop fan-out on a huge index hammers your vector DB; bound
    total retrievals, not just hops.

## 9 · Verification checklist

Deterministic checks proving every claim in this lesson. All should print **PASS**.

In [ ]:
def check(name, cond):
    print(f"[{'PASS' if cond else 'FAIL'}] {name}")
    return cond

ok = True
# 1. The bridge question is genuinely unreachable single-shot.
ok &= check("single-shot cannot reach gold p2 for the bridge question",
            "p2" not in [p for p, _, _ in retrieve(BRIDGE_Q, k=3)])
# 2. Agentic RAG DOES reach it.
res = agentic_rag(BRIDGE_Q, verbose=False)
ok &= check("agentic RAG reaches gold p2 via a hop", "p2" in res["evidence_ids"])
# 3. It actually looped.
ok &= check("agentic RAG used >1 hop on the bridge question", len(res["trace"]) > 1)
# 4. The bridge term 'titan-x' was mined from p1 and drove the next hop.
ok &= check("bridge term 'titan-x' mined from p1", "titan-x" in bridge_terms(BRIDGE_Q, [KB["p1"]]))
# 5. Hop budget respected even on an unanswerable question.
ok &= check("hop budget never exceeded",
            len(agentic_rag("unrelated quantum chromodynamics question", verbose=False)["trace"]) <= 3)
# 6. CRAG abstains on a truly out-of-scope question.
ok &= check("CRAG abstains on out-of-scope query",
            corrective_rag("What is the capital of France?").get("abstained") is True)
# 7. The loop keeps an easy lookup to a single hop.
ok &= check("easy lookup terminates in 1 hop",
            routed_rag("What portafilter does the Barista Pro use?")["route"] == "single-shot")
# 8. The loop escalates the bridge question.
ok &= check("bridge question escalates to agentic",
            routed_rag(BRIDGE_Q)["route"] == "agentic")
# 9. Agentic never does worse than single-shot across the eval set.
ss_h = sum(single_shot_hit1(q, g) for q, g, _ in EVAL)
ag_h = sum(agentic_eval(q, g)[0] for q, g, _ in EVAL)
ok &= check(f"agentic gold-reach ({ag_h}) >= single-shot hit@1 ({ss_h})", ag_h >= ss_h)

print("\nALL CHECKS PASS" if ok else "\nSOME CHECKS FAILED")


## 10 · Summary, homework & what's next

### What you built
A real **agentic RAG loop** — retrieve → **grade your own evidence** → mine bridge terms → hop again
→ grounded, cited answer — plus **CRAG** (grade the retrieval, rewrite-or-abstain) and a grader that
**self-routes** (easy questions terminate in 1 hop). And you **measured** the uplift: it lands
exactly on the multi-hop questions single-shot RAG structurally cannot reach.

### The one idea to keep
> **Single-shot RAG retrieves once and hopes. Agentic RAG retrieves, judges its own evidence, and
> keeps going — under a strict hop budget — until it can actually answer.** The loop, the grader,
> and the budget matter more than any single clever call.

### How this connects
- **L4 (ReAct)** — this *is* the ReAct loop, pointed at retrieval instead of tools.
- **L88 (query transformation)** — each hop's rewrite is a transformation, now triggered
  *conditionally* by the grader.
- **L86/L87** — a production hop is a full pipeline: hybrid retrieve → rerank → grounded answer.
- **L29** — the grader is a classifier; calibrate it with Brier/ECE.

### Homework
1. **Real hops.** Add an `ANTHROPIC_API_KEY` and re-run. Compare the LLM's next-hop queries and
   grade verdicts against the deterministic proxies. Where does it stop earlier or hop smarter?
2. **Full pipeline per hop.** Replace the bare `retrieve()` inside the loop with
   `transform → hybrid → rerank` from L85/L86/L88. Re-run the eval — how much does bridge accuracy move?
3. **A three-hop chain.** Add a two-bridge question (machine → boiler → *manufacturer* → warranty)
   that needs **3** hops, and confirm the loop walks the whole chain within budget.
4. **Calibrate the grader.** Build a 10-question labeled set of (question, is-sufficient) pairs and
   measure your grader's ECE. Does it stop too early or too late?
5. **Cost dashboard.** Log retrievals **and** LLM calls per question, and plot accuracy vs. cost for
   single-shot / routed / always-agentic.

### What's next — Lesson 90
**Phase 10 Capstone — ship a production RAG service.** We fold the whole phase (chunking → hybrid →
rerank → grounding → transformation → this agentic loop) into one `rag-service`: a FastAPI `/ask`
endpoint with the self-routing loop, an eval-gated CI, and a small open-source repo you can point
to. The retrieval half of your portfolio, shipped.

*Curious about something today? Drop your question on the next scheduled run — questions reshape the
curriculum.*